In [1]:
!pip install chemprop rdkit -q

In [4]:
import os, json, shutil, inspect, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch.serialization
import lightning.pytorch as pl
from rdkit import Chem
from scipy.special import expit, logit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, matthews_corrcoef, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix, brier_score_loss
)
import chemprop
from chemprop import data, featurizers, models
from chemprop import nn as cpnn
from chemprop.nn import BCELoss as ChempropBCELoss

V1_CKPT   = "/kaggle/input/datasets/amirtesh/ad-rl-chemprop-model/AD_MTDL_best.ckpt"
DATA_PATH = "/kaggle/input/datasets/amirtesh/alzheimer-targets/AD_MTDL_combined_chembl36.csv"
SAVE_DIR  = "/kaggle/working/AD_MTDL_v1ckpt_corrected_eval"
V2_SPLIT_JSON = "/kaggle/working/AD_MTDL_model_v2/split_indices.json"  
TARGET_COLS = ["AChE_class", "BuChE_class", "BACE1_class", "MAO_B_class"]
RANDOM_SEED = 42
EXPECTED_SPLIT = (9039, 1129, 1129)
N_BOOT = 1000
LOGIT_GRID = np.round(np.arange(-12.0, 12.0001, 0.05), 2)   
PLATEAU_TOL = 0.005                                          

V1_THR = {"AChE_class": 0.59, "BuChE_class": 0.68, "BACE1_class": 0.60, "MAO_B_class": 0.51}
V1_CM = {  
    ("AChE_class", "val"):  (85, 8, 11, 92),    ("AChE_class", "test"):  (97, 14, 27, 168),
    ("BuChE_class", "val"): (58, 2, 3, 59),     ("BuChE_class", "test"): (71, 7, 3, 92),
    ("BACE1_class", "val"): (64, 11, 9, 470),   ("BACE1_class", "test"): (48, 9, 11, 463),
    ("MAO_B_class", "val"): (143, 28, 29, 102), ("MAO_B_class", "test"): (74, 16, 28, 69),
}
V1_TEST_AUROC = {"AChE_class": 0.9281, "BuChE_class": 0.9651, "BACE1_class": 0.9856, "MAO_B_class": 0.8532}

assert not V1_CKPT.startswith("PASTE"), "Set V1_CKPT."
os.makedirs(SAVE_DIR, exist_ok=True)
pl.seed_everything(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("chemprop", chemprop.__version__, "| device", device)

df = pd.read_csv(DATA_PATH)
class_mapping = {"Active": 1, "Inactive": 0, "Moderate": np.nan}
for col in TARGET_COLS:
    df[col] = df[col].map(class_mapping)
df = df.dropna(subset=TARGET_COLS, how="all").reset_index(drop=True)
df = df[df["smiles"].apply(lambda s: Chem.MolFromSmiles(str(s)) is not None)].reset_index(drop=True)
print(f"Compounds after cleaning: {len(df):,}")
for col in TARGET_COLS:
    print(f"  {col:<13}: Active={(df[col]==1).sum():>5}  Inactive={(df[col]==0).sum():>5}  NaN={df[col].isna().sum():>5}")

targets_array = df[TARGET_COLS].to_numpy(dtype=np.float32)
datapoints = [data.MoleculeDatapoint.from_smi(s, y) for s, y in zip(df["smiles"].tolist(), targets_array)]
mols = [d.mol for d in datapoints]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    train_idx, val_idx, test_idx = data.make_split_indices(mols, split="scaffold_balanced", sizes=(0.8, 0.1, 0.1))
train_data, val_data, test_data = data.split_data_by_indices(datapoints, train_idx, val_idx, test_idx)
train_data, val_data, test_data = train_data[0], val_data[0], test_data[0]
got = (len(train_data), len(val_data), len(test_data))
assert got == EXPECTED_SPLIT, f"Split {got} != v1 {EXPECTED_SPLIT}"
split_indices = {"train": [int(i) for i in train_idx[0]], "val": [int(i) for i in val_idx[0]], "test": [int(i) for i in test_idx[0]]}
if os.path.exists(V2_SPLIT_JSON):
    assert json.load(open(V2_SPLIT_JSON)) == split_indices, "Split differs from saved v2 split_indices.json"
    print("Split identical to saved v2 split_indices.json")
json.dump(split_indices, open(os.path.join(SAVE_DIR, "split_indices.json"), "w"))
df.to_csv(os.path.join(SAVE_DIR, "dataset_after_cleaning.csv"), index=False)

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
val_loader  = data.build_dataloader(data.MoleculeDataset(val_data,  featurizer), shuffle=False, num_workers=2)
test_loader = data.build_dataloader(data.MoleculeDataset(test_data, featurizer), shuffle=False, num_workers=2)

fwd_src = inspect.getsource(cpnn.BinaryClassificationFFN.forward)
trn_src = inspect.getsource(cpnn.BinaryClassificationFFN.train_step)
print("\n--- BinaryClassificationFFN.forward ---\n" + fwd_src + "--- BinaryClassificationFFN.train_step ---\n" + trn_src)
assert "sigmoid" in fwd_src, "forward() does not apply sigmoid; assumptions of this script are violated."
assert "sigmoid" not in trn_src, "train_step() applies sigmoid; it would not return logits."

class WeightedBCELoss(ChempropBCELoss):  
    def __init__(self, pos_weight):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight)
    def _calc_unreduced_loss(self, preds, targets, *args, **kwargs):
        return F.binary_cross_entropy_with_logits(preds, targets, reduction="none",
                                                  pos_weight=self.pos_weight.to(preds.device))

torch.serialization.add_safe_globals([WeightedBCELoss, cpnn.metrics.BinaryAUPRC, cpnn.metrics.BinaryAUROC])
best_model = models.MPNN.load_from_checkpoint(V1_CKPT).to(device).eval()
print(f"\nLoaded v1 checkpoint: {V1_CKPT} | params={sum(p.numel() for p in best_model.parameters()):,}")

def get_outputs(loader):
    P, L, T = [], [], []
    with torch.no_grad():
        for batch in loader:
            bmg, X_vd, features, targets, weights, lt_mask, gt_mask = batch
            bmg.V = bmg.V.to(device); bmg.E = bmg.E.to(device)
            bmg.edge_index = bmg.edge_index.to(device); bmg.batch = bmg.batch.to(device)
            X_vd = X_vd.to(device) if X_vd is not None else None
            features = [f.to(device) for f in features] if features else features
            Z = best_model.fingerprint(bmg, X_vd, features)
            L.append(best_model.predictor.train_step(Z).cpu().numpy())   
            P.append(best_model.predictor(Z).cpu().numpy())             
            T.append(targets.numpy())
    return np.vstack(P), np.vstack(L).astype(np.float64), np.vstack(T)

val_P,  val_L,  val_T  = get_outputs(val_loader)
test_P, test_L, test_T = get_outputs(test_loader)

print("\n===== GATE 0: logits consistent with forward() probabilities; no compression =====")
for nm, P, L in [("val", val_P, val_L), ("test", test_P, test_L)]:
    err = float(np.max(np.abs(expit(L) - P)))
    print(f"  {nm}: max|expit(logit)-forward|={err:.2e} | prob range [{P.min():.4f}, {P.max():.4f}] | logit range [{L.min():.2f}, {L.max():.2f}]"
          f" | exact 0/1 outputs: {int((P==0).sum())}/{int((P==1).sum())} of {P.size}")
    assert err < 1e-5, "train_step logits do not match forward() probabilities."
    assert P.min() >= 0 and P.max() <= 1 and P.max() > 0.9, "Probabilities look compressed; a second sigmoid may still exist."
print("GATE 0 PASSED. NOTE: float32 probabilities saturate at exactly 0/1 (ties); all ranking/threshold work below uses logits.")

print("\n===== GATE 1: v1 confusion-matrix reproduction (rule: expit(p) >= v1_threshold, p = forward() prob) =====")
all_match = True
for split, P, T in [("val", val_P, val_T), ("test", test_P, test_T)]:
    for t, col in enumerate(TARGET_COLS):
        m = ~np.isnan(T[:, t]); y = T[m, t].astype(int); p = P[m, t]
        cm = tuple(int(x) for x in confusion_matrix(y, (expit(p) >= V1_THR[col]).astype(int), labels=[0, 1]).ravel())
        ok = cm == V1_CM[(col, split)]; all_match &= ok
        print(f"  {col:<12} {split:<4} cm={cm}  {'MATCH' if ok else 'MISMATCH (v1 was ' + str(V1_CM[(col, split)]) + ')'}")
assert all_match, "GATE 1 FAILED: v1 results not reproduced. Do not proceed."
print("GATE 1 PASSED: v1 tables reproduced; split, data and checkpoint identical to the submitted run.")
print("\nv1 thresholds expressed correctly (v1 'prob' threshold t applied to sigmoid(p)  ==  p >= logit(t); in logit space that is exactly t):")
for col in TARGET_COLS:
    print(f"  {col:<12} v1 {V1_THR[col]:.2f} on sigmoid(p)  ==  p >= {logit(V1_THR[col]):.4f}  ==  logit >= {logit(logit(V1_THR[col])) if 0 < logit(V1_THR[col]) < 1 else float('nan'):.3f}"
          if 0 < logit(V1_THR[col]) < 1 else f"  {col:<12} v1 {V1_THR[col]:.2f}  ==  p >= {logit(V1_THR[col]):.4f}")

print("\n===== GATE 2: AUROC — v1 method reproduced, then logit-based (tie-free) =====")
test_P_v1way = torch.sigmoid(torch.tensor(test_P, device=device)).cpu().numpy()   
for t, col in enumerate(TARGET_COLS):
    m = ~np.isnan(test_T[:, t]); y = test_T[m, t].astype(int)
    a_v1way = roc_auc_score(y, test_P_v1way[m, t]); a_prob = roc_auc_score(y, test_P[m, t]); a_logit = roc_auc_score(y, test_L[m, t])
    flag = "v1 REPRODUCED" if abs(a_v1way - V1_TEST_AUROC[col]) < 2e-4 else "v1 not bit-exact (float32/device ties; Gate 1 already proves identical decisions)"
    print(f"  {col:<12} v1 {V1_TEST_AUROC[col]:.4f} | v1-method {a_v1way:.4f} ({flag}) | float32 prob {a_prob:.4f} | logit {a_logit:.4f}")
    assert abs(a_logit - V1_TEST_AUROC[col]) < 0.01, f"Logit AUROC for {col} differs from v1 by >0.01; investigate."
print("GATE 2 PASSED: new AUROC (from logits) within 0.01 of v1; differences are float32 tie artifacts, not a model change.")

def counts(y, pred):
    tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    return tn, fp, fn, tp
def mcc_counts(tn, fp, fn, tp):
    den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return (tp * tn - fp * fn) / den if den > 0 else 0.0

optimal_thr_logit, optimal_thr_prob, sweep_rows = {}, {}, []
print("\n===== MCC THRESHOLD SWEEP (val, logit grid -12..12 step 0.05) =====")
for t, col in enumerate(TARGET_COLS):
    m = ~np.isnan(val_T[:, t]); y = val_T[m, t].astype(int); s = val_L[m, t]
    mccs = np.array([mcc_counts(*counts(y, (s >= thr).astype(int))) for thr in LOGIT_GRID])
    for thr, mc in zip(LOGIT_GRID, mccs):
        sweep_rows.append({"Task": col, "Threshold_logit": thr, "Threshold_prob": float(expit(thr)), "MCC": float(mc)})
    i_best = int(np.argmax(mccs)); lo = hi = i_best
    while lo > 0 and mccs[lo - 1] >= mccs[i_best] - PLATEAU_TOL: lo -= 1
    while hi < len(LOGIT_GRID) - 1 and mccs[hi + 1] >= mccs[i_best] - PLATEAU_TOL: hi += 1
    i_sel = (lo + hi) // 2
    optimal_thr_logit[col] = float(LOGIT_GRID[i_sel]); optimal_thr_prob[col] = float(expit(LOGIT_GRID[i_sel]))
    print(f"  {col:<12} exact-max logit {LOGIT_GRID[i_best]:+.2f} (p={expit(LOGIT_GRID[i_best]):.3f}, MCC {mccs[i_best]:.4f}) | "
          f"plateau logit [{LOGIT_GRID[lo]:+.2f}, {LOGIT_GRID[hi]:+.2f}] (p [{expit(LOGIT_GRID[lo]):.3f}, {expit(LOGIT_GRID[hi]):.3f}]) | "
          f"SELECTED logit {LOGIT_GRID[i_sel]:+.2f} (p={expit(LOGIT_GRID[i_sel]):.3f}, val MCC {mccs[i_sel]:.4f})")
pd.DataFrame(sweep_rows).to_csv(os.path.join(SAVE_DIR, "mcc_sweep_curves_val.csv"), index=False)

def evaluate(L, T, thr_logit, split):
    rows = []
    for t, col in enumerate(TARGET_COLS):
        m = ~np.isnan(T[:, t]); y = T[m, t].astype(int); s = L[m, t]; p = expit(s)
        yp = (s >= thr_logit[col]).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()
        rows.append({
            "Task": col, "Split": split, "N": int(m.sum()),
            "Thr_logit": round(thr_logit[col], 2), "Thr_prob": round(float(expit(thr_logit[col])), 4),
            "AUROC": round(roc_auc_score(y, s), 4), "AUPRC": round(average_precision_score(y, s), 4),
            "MCC": round(matthews_corrcoef(y, yp), 4), "Accuracy": round(accuracy_score(y, yp), 4),
            "Sens": round(tp / max(tp + fn, 1), 4), "Spec": round(tn / max(tn + fp, 1), 4),
            "F1_W": round(f1_score(y, yp, average="weighted", zero_division=0), 4),
            "F1_Mac": round(f1_score(y, yp, average="macro", zero_division=0), 4),
            "Prec_Pos": round(precision_score(y, yp, zero_division=0), 4),
            "Rec_Pos": round(recall_score(y, yp, zero_division=0), 4),
            "F1_Pos": round(f1_score(y, yp, zero_division=0), 4),
            "Brier": round(brier_score_loss(y, p), 4), "Prevalence": round(float(y.mean()), 4),
            "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        })
    return pd.DataFrame(rows).set_index(["Task", "Split"])

pd.set_option("display.max_columns", None); pd.set_option("display.width", 260)
pd.set_option("display.float_format", "{:.4f}".format)
all_results = pd.concat([evaluate(val_L, val_T, optimal_thr_logit, "val"),
                         evaluate(test_L, test_T, optimal_thr_logit, "test")]).sort_index(level="Task")
print("\n===== FULL METRICS - VAL + TEST =====\n")
print(all_results.drop(columns=["TP", "FP", "TN", "FN"]).to_string())
print("\n===== CONFUSION MATRICES =====")
for (task, split), r in all_results.iterrows():
    print(f"\n  {task} [{split}] (logit thr={r.Thr_logit}, p={r.Thr_prob})\n              Pred 0  Pred 1")
    print(f"  Actual 0   {int(r.TN):>6}  {int(r.FP):>6}\n  Actual 1   {int(r.FN):>6}  {int(r.TP):>6}")
test_res = all_results.xs("test", level="Split")

def bootstrap_ci(y, s, thr, n=N_BOOT, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed); keys = ["AUROC", "AUPRC", "MCC", "Sens", "Spec"]
    S = {k: [] for k in keys}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for _ in range(n):
            i = rng.integers(0, len(y), len(y)); yy, ss = y[i], s[i]
            if yy.min() == yy.max(): continue
            yp = (ss >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(yy, yp, labels=[0, 1]).ravel()
            S["AUROC"].append(roc_auc_score(yy, ss)); S["AUPRC"].append(average_precision_score(yy, ss))
            S["MCC"].append(matthews_corrcoef(yy, yp)); S["Sens"].append(tp / max(tp + fn, 1)); S["Spec"].append(tn / max(tn + fp, 1))
    return {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for k, v in S.items()}

ci_rows, calib_rows = [], []
for t, col in enumerate(TARGET_COLS):
    m = ~np.isnan(test_T[:, t]); y = test_T[m, t].astype(int); s = test_L[m, t]; p = expit(s)
    ci = bootstrap_ci(y, s, optimal_thr_logit[col]); pt = test_res.loc[col]
    row = {"Task": col, "N": int(m.sum()), **{k: round(float(pt[k]), 4) for k in ["AUROC", "AUPRC", "MCC", "Sens", "Spec"]},
           **{f"{k}_CI95": f"[{lo:.3f}, {hi:.3f}]" for k, (lo, hi) in ci.items()}}
    edges = np.linspace(0, 1, 11); ece = 0.0
    for b in range(10):
        sel = (p >= edges[b]) & ((p < edges[b + 1]) if b < 9 else (p <= edges[b + 1]))
        if sel.sum() == 0: continue
        calib_rows.append({"Task": col, "bin_lo": edges[b], "bin_hi": edges[b + 1], "n": int(sel.sum()),
                           "mean_pred": float(p[sel].mean()), "frac_pos": float(y[sel].mean())})
        ece += sel.sum() / len(p) * abs(p[sel].mean() - y[sel].mean())
    row["ECE_10bin"] = round(float(ece), 4); row["Brier"] = float(pt["Brier"]); row["Prevalence"] = float(pt["Prevalence"])
    ci_rows.append(row)
ci_df = pd.DataFrame(ci_rows); calib_df = pd.DataFrame(calib_rows)
print(f"\n===== TEST METRICS WITH BOOTSTRAP 95% CIs (n_boot={N_BOOT}), ECE, Brier =====\n")
print(ci_df.to_string(index=False))
print("\nNOTE: training used per-task pos_weight, so outputs are class-balanced scores, not prevalence-calibrated probabilities, "
      "and the network saturates at 0/1. Report ECE/Brier honestly; calibrate post hoc on validation (e.g., Platt scaling on logits) "
      "before using probabilities as RL rewards.")

for nm, arr in [("val_probs_float32", val_P), ("val_logits", val_L), ("val_targets", val_T),
                ("test_probs_float32", test_P), ("test_logits", test_L), ("test_targets", test_T)]:
    np.save(os.path.join(SAVE_DIR, f"{nm}.npy"), arr)
shutil.copy2(V1_CKPT, os.path.join(SAVE_DIR, "AD_MTDL_v1_best.ckpt"))
all_results.reset_index().to_csv(os.path.join(SAVE_DIR, "AD_MTDL_results_corrected.csv"), index=False)
ci_df.to_csv(os.path.join(SAVE_DIR, "AD_MTDL_test_metrics_with_CI.csv"), index=False)
calib_df.to_csv(os.path.join(SAVE_DIR, "AD_MTDL_calibration_bins_test.csv"), index=False)
json.dump({"thresholds_logit": optimal_thr_logit, "thresholds_probability": optimal_thr_prob,
           "rule": f"MCC sweep on validation in logit space, centre of contiguous plateau within {PLATEAU_TOL} of max MCC",
           "v1_thresholds_applied_to_sigmoid_of_prob": V1_THR},
          open(os.path.join(SAVE_DIR, "thresholds.json"), "w"), indent=2)
json.dump({
    "project": "AD-MTDL multi-task ChemProp classifier - v1 checkpoint, corrected inference",
    "created": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "chemprop_version": chemprop.__version__,
    "random_seed": RANDOM_SEED, "targets": TARGET_COLS,
    "checkpoint": "AD_MTDL_v1_best.ckpt (unchanged v1 weights; no retraining)",
    "training_note": "predictor.train_step returns raw logits and is what the loss consumed; v1 training was correct (verified from source and by reproducing v1 confusion matrices).",
    "inference_note": "forward() already applies sigmoid; v1 evaluation applied a second sigmoid (range [0.5, 0.731]). Corrected pipeline scores on raw logits (train_step) and derives probabilities by a single sigmoid in float64.",
    "gates": {"gate0_logits_consistent": True, "gate1_v1_confusion_matrices_reproduced": True, "gate2_logit_auroc_within_0.01_of_v1": True},
    "split": {"method": "scaffold_balanced", "sizes": [0.8, 0.1, 0.1], "train_n": got[0], "val_n": got[1], "test_n": got[2], "indices_file": "split_indices.json"},
    "threshold_optimization": {"space": "logit", "grid": "-12..12 step 0.05", "plateau_tolerance": PLATEAU_TOL,
                               "thresholds_logit": optimal_thr_logit, "thresholds_probability": optimal_thr_prob},
    "bootstrap": {"n": N_BOOT, "seed": RANDOM_SEED, "level": 0.95},
}, open(os.path.join(SAVE_DIR, "AD_MTDL_corrected_metadata.json"), "w"), indent=2)
shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)
print(f"\nZip ready: {SAVE_DIR}.zip")
for fn in sorted(os.listdir(SAVE_DIR)):
    print(f"  {fn:<44} {os.path.getsize(os.path.join(SAVE_DIR, fn))/1024:.1f} KB")
print("\nALL GATES PASSED. Use thresholds.json (logit + probability) for external validation and the scoring server.")

Seed set to 42


chemprop 2.3.1 | device cuda
Compounds after cleaning: 11,297
  AChE_class   : Active= 1536  Inactive= 1164  NaN= 8597
  BuChE_class  : Active=  939  Inactive=  761  NaN= 9597
  BACE1_class  : Active= 4739  Inactive=  750  NaN= 5808
  MAO_B_class  : Active=  978  Inactive= 1135  NaN= 9184


The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)



--- BinaryClassificationFFN.forward ---
    def forward(self, Z: Tensor) -> Tensor:
        Y = super().forward(Z)

        return Y.sigmoid()
--- BinaryClassificationFFN.train_step ---
    def train_step(self, Z: Tensor) -> Tensor:
        return super().forward(Z)


Loaded v1 checkpoint: /kaggle/input/datasets/amirtesh/ad-rl-chemprop-model/AD_MTDL_best.ckpt | params=410,104

===== GATE 0: logits consistent with forward() probabilities; no compression =====
  val: max|expit(logit)-forward|=8.45e-08 | prob range [0.0000, 1.0000] | logit range [-45.62, 41.85] | exact 0/1 outputs: 0/308 of 4516
  test: max|expit(logit)-forward|=8.56e-08 | prob range [0.0000, 1.0000] | logit range [-57.28, 32.85] | exact 0/1 outputs: 0/272 of 4516
GATE 0 PASSED. NOTE: float32 probabilities saturate at exactly 0/1 (ties); all ranking/threshold work below uses logits.

===== GATE 1: v1 confusion-matrix reproduction (rule: expit(p) >= v1_threshold, p = forward() prob) =====
  AChE_class   val  cm=(85, 8, 11